# spectral-watermark: Beginner-Friendly API Reference

This guide documents every public Python class and function in the package, plus the complete command-line interface. Examples use tiny in-memory arrays and print only simple shapes and values.

## 1. Package Overview & Quick Map

| Module | Plain-English purpose |
|---|---|
| `pipeline` | Provides the easy end-to-end pipeline for embedding and finding messages in grayscale, RGB, or RGBA images. |
| `ecc` | Converts text to protected bit packets and tries to repair small saved-image errors. |
| `core` | Works with one 64 × 64 tile and changes a few pattern strengths in that tile. |
| `calibration` | Finds matching pilot patterns and trims plain screenshot borders. |
| `cli` | Runs embedding, extraction, and distortion tests from a terminal. |

**Start here:** 95% of users should begin with `WatermarkPipeline`.

## 2. High-Level Pipeline (`pipeline.py`)

`WatermarkPipeline` combines text protection, tile embedding, color handling, border trimming, and crop search.

### WatermarkPipeline.__init__

**Signature:** `WatermarkPipeline(block_size=64, r_min=8.0, r_max=22.0, seed=2026)`

**What it does:** Creates a pipeline and its text codec, tile engine, and scale calibrator.

#### Parameters

| Parameter | Accepted Type(s) | Default | Simple Explanation |
|---|---|---|---|
| `block_size` | `int` | `64` | Width and height of each square tile. |
| `r_min` | `float` | `8.0` | Inner edge of the pattern ring. |
| `r_max` | `float` | `22.0` | Outer edge of the pattern ring. |
| `seed` | `int` | `2026` | Makes pattern choice repeatable.|

**Returns:** A `WatermarkPipeline` object. No value is returned explicitly.

In [1]:
from spectral_watermark import WatermarkPipeline
import numpy as np
p = WatermarkPipeline()
print(type(p).__name__)
print(p.engine.block_size, p.engine.r_min, p.engine.r_max)

WatermarkPipeline
64 8.0 22.0


### WatermarkPipeline.embed_image

**Signature:** `embed_image(image, payload, alpha=0.45)`

**What it does:** Adds an encoded text payload to a grayscale, RGB, or RGBA image.

#### Parameters

| Parameter | Accepted Type(s) | Default | Simple Explanation |
|---|---|---|---|
| `image` | `np.ndarray \| PIL.Image.Image` | Required | The cover image to modify. |
| `payload` | `str` | Required | UTF-8 text to hide. |
| `alpha` | `float` | `0.45` | How strongly the watermark changes patterns.|

**Returns:** A tuple `(np.ndarray, int)`: the watermarked image, and the number of encoded payload bits.

In [2]:
import numpy as np
from spectral_watermark import WatermarkPipeline
cover = np.zeros((64, 64), dtype=np.uint8)
stego, bit_len = WatermarkPipeline().embed_image(cover, 'HI')
print(stego.shape, stego.dtype, bit_len)

(64, 64) uint8 104


### WatermarkPipeline.extract_from_crop

**Signature:** `extract_from_crop(cropped_image, expected_bit_len=None, search_stride=2, scales=None, min_chars=1, max_chars=32, on_progress=None, timeout=None, on_scale_detected=None)`

**What it does:** Searches a full image or crop for a hidden text payload, including common offsets and resized views.

#### Parameters

| Parameter | Accepted Type(s) | Default | Simple Explanation |
|---|---|---|---|
| `cropped_image` | `np.ndarray \| PIL.Image.Image` | Required | Full image or crop to search. |
| `expected_bit_len` | `int \| None` | `None` | Known encoded bit length. |
| `search_stride` | `int` | `2` | Pixel spacing in the detailed search. |
| `scales` | `list[float] \| None` | `None` | Extra zoom factors to try. |
| `min_chars` | `int` | `1` | Smallest allowed text length. |
| `max_chars` | `int` | `32` | Largest allowed text length. |
| `on_progress` | `Callable[[int, int, int, int, int], None] \| None` | `None` | Optional progress callback. |
| `timeout` | `float \| None` | `None` | Maximum search time in seconds. |
| `on_scale_detected` | `Callable[[float], None] \| None` | `None` | Optional detected-scale callback.|

**Returns:** `str \| None`: the recovered text, or `None` when no valid payload is found.

In [3]:
import numpy as np
from spectral_watermark import WatermarkPipeline
p = WatermarkPipeline()
stego, n = p.embed_image(np.zeros((64, 64), dtype=np.uint8), 'HI')
text = p.extract_from_crop(stego, expected_bit_len=n, timeout=2.0)
print(type(text).__name__, text)

NoneType None


## 3. Text-to-Bits & Error Protection (`ecc.py`)

`BitCodec` adds a short start marker and backup check bytes. It can repair a moderate number of wrong values caused by saving or image changes.

### BitCodec.__init__

**Signature:** `BitCodec(parity_bytes=8, sync_marker=None)`

**What it does:** Creates a text packet encoder and decoder with configurable backup check bytes.

#### Parameters

| Parameter | Accepted Type(s) | Default | Simple Explanation |
|---|---|---|---|
| `parity_bytes` | `int` | `8` | Number of backup check bytes; must be positive. |
| `sync_marker` | `bytes \| None` | `None` | Custom start marker; default is `b'Z\\xa5'`.|

**Returns:** A `BitCodec` object. No value is returned explicitly.

In [4]:
from spectral_watermark import BitCodec
codec = BitCodec()
print(codec.parity_bytes, codec.sync_marker.hex())

8 5aa5


### BitCodec.encode_payload

**Signature:** `encode_payload(text)`

**What it does:** Turns a string into a marked, length-framed, error-protected stream of 0 and 1 values.

#### Parameters

| Parameter | Accepted Type(s) | Default | Simple Explanation |
|---|---|---|---|
| `text` | `str` | Required | UTF-8 text to encode; encoded size must not exceed 255 bytes.|

**Returns:** A 1D `numpy.ndarray` with dtype `uint8`, containing 0 and 1 values.

In [5]:
from spectral_watermark import BitCodec
bits = BitCodec().encode_payload('Hi')
print(type(bits).__name__, bits.shape, bits.dtype, len(bits))

ndarray (104,) uint8 104


### BitCodec.decode_packet

**Signature:** `decode_packet(bitstream)`

**What it does:** Checks a bit stream's start marker, repairs backup bytes, validates framing, and reads UTF-8 text.

#### Parameters

| Parameter | Accepted Type(s) | Default | Simple Explanation |
|---|---|---|---|
| `bitstream` | `np.ndarray` | Required | 1D or reshapeable values containing packet bits.|

**Returns:** `str \| None`: the recovered text, or `None` if the marker or error protection fails.

In [6]:
from spectral_watermark import BitCodec
codec = BitCodec()
bits = codec.encode_payload('Hi')
print(codec.decode_packet(bits))
print(codec.decode_packet(bits[:16]))

Hi
None


## 4. 64×64 Tile Watermark Engine (`core.py`)

`SpectralEngine` hides and reads binary values inside pattern strengths in one tile. It does not know text or error protection.

### SpectralEngine.__init__

**Signature:** `SpectralEngine(block_size=64, r_min=8.0, r_max=22.0, seed=2026)`

**What it does:** Creates a tile engine and validates that tile dimensions and pattern settings are usable.

#### Parameters

| Parameter | Accepted Type(s) | Default | Simple Explanation |
|---|---|---|---|
| `block_size` | `int` | `64` | Positive even tile width and height. |
| `r_min` | `float` | `8.0` | Inner pattern-ring radius. |
| `r_max` | `float` | `22.0` | Outer pattern-ring radius. |
| `seed` | `int` | `2026` | Seed for repeatable coordinate choice.|

**Returns:** A `SpectralEngine` object. No value is returned explicitly.

In [7]:
from spectral_watermark import SpectralEngine
engine = SpectralEngine()
print(type(engine).__name__, engine.block_size)

SpectralEngine 64


### SpectralEngine.get_bin_pairs

**Signature:** `get_bin_pairs(num_bits)`

**What it does:** Finds two distinct pattern positions for every bit, with pilot positions left unused.

#### Parameters

| Parameter | Accepted Type(s) | Default | Simple Explanation |
|---|---|---|---|
| `num_bits` | `int` | Required | Positive number of bits to allocate positions for.|

**Returns:** A tuple `(list[tuple[int, int]], list[tuple[int, int]])` containing first and second tile coordinates.

In [8]:
from spectral_watermark import SpectralEngine
a, b = SpectralEngine().get_bin_pairs(4)
print(len(a), len(b), type(a).__name__, type(b).__name__)

4 4 list list


### SpectralEngine.get_bin_arrays

**Signature:** `get_bin_arrays(num_bits)`

**What it does:** Gets the same coordinate pairs as arrays, caching them for repeated crop scans.

#### Parameters

| Parameter | Accepted Type(s) | Default | Simple Explanation |
|---|---|---|---|
| `num_bits` | `int` | Required | Positive number of bits to allocate.|

**Returns:** A tuple `(np.ndarray, np.ndarray)`, both shape `(num_bits, 2)` with dtype `intp`.

In [9]:
from spectral_watermark import SpectralEngine
a, b = SpectralEngine().get_bin_arrays(4)
print(a.shape, b.shape, a.dtype)

(4, 2) (4, 2) int64


### SpectralEngine.embed_tile

**Signature:** `embed_tile(tile, bits, pairs_a, pairs_b, alpha=0.45)`

**What it does:** Changes pattern strengths in a 64×64 tile to represent 0 and 1 values.

#### Parameters

| Parameter | Accepted Type(s) | Default | Simple Explanation |
|---|---|---|---|
| `tile` | `np.ndarray` | Required | Square `64×64` grayscale pixel values. |
| `bits` | `np.ndarray` | Required | 1D bit values to embed. |
| `pairs_a` | `list[tuple[int, int]]` | Required | First pattern position for each bit. |
| `pairs_b` | `list[tuple[int, int]]` | Required | Second pattern position for each bit. |
| `alpha` | `float` | `0.45` | Strength multiplier for the change.|

**Returns:** A clipped `np.ndarray` shape `(64, 64)` with dtype `float64`, containing the watermarked tile.

In [10]:
import numpy as np
from spectral_watermark import SpectralEngine
e = SpectralEngine(); a, b = e.get_bin_pairs(4)
tile = e.embed_tile(np.zeros((64, 64)), np.array([1,0,1,0]), a, b)
print(tile.shape, tile.dtype, float(tile.min()), float(tile.max()))

(64, 64) float64 0.0 0.030859375


### SpectralEngine.extract_bits

**Signature:** `extract_bits(tile, pairs_a, pairs_b)`

**What it does:** Compares each pair of pattern strengths and reads each result as 0 or 1.

#### Parameters

| Parameter | Accepted Type(s) | Default | Simple Explanation |
|---|---|---|---|
| `tile` | `np.ndarray` | Required | Square `64×64` grayscale tile. |
| `pairs_a` | `list[tuple[int, int]]` | Required | First coordinate for each pair. |
| `pairs_b` | `list[tuple[int, int]]` | Required | Second coordinate for each pair.|

**Returns:** A 1D `np.ndarray` with dtype `uint8`, containing extracted 0 and 1 values.

In [11]:
import numpy as np
from spectral_watermark import SpectralEngine
e = SpectralEngine(); a, b = e.get_bin_pairs(4)
tile = e.embed_tile(np.zeros((64,64)), np.array([1,0,1,0]), a, b)
bits = e.extract_bits(tile, a, b)
print(bits.shape, bits.dtype, bits.tolist())

(4,) uint8 [1, 0, 1, 0]


### SpectralEngine.extract_bits_from_spectrum

**Signature:** `extract_bits_from_spectrum(spectrum, pairs_a, pairs_b)`

**What it does:** Reads bit values directly from an already computed 2D real FFT spectrum.

#### Parameters

| Parameter | Accepted Type(s) | Default | Simple Explanation |
|---|---|---|---|
| `spectrum` | `np.ndarray` | Required | 2D complex real-FFT result. |
| `pairs_a` | `list[tuple[int, int]]` | Required | First coordinate for each pair. |
| `pairs_b` | `list[tuple[int, int]]` | Required | Second coordinate for each pair.|

**Returns:** A 1D `np.ndarray` with dtype `uint8`, containing extracted 0 and 1 values.

In [12]:
import numpy as np
from spectral_watermark import SpectralEngine
e = SpectralEngine(); a, b = e.get_bin_pairs(4)
spectrum = np.fft.rfft2(np.zeros((64,64)))
bits = e.extract_bits_from_spectrum(spectrum, a, b)
print(bits.shape, bits.dtype, bits.tolist())

(4,) uint8 [0, 0, 0, 0]


### SpectralEngine.check_sync_preamble

**Signature:** `check_sync_preamble(spectrum, pairs_a, pairs_b, sync_marker)`

**What it does:** Quickly checks whether the spectrum's first 16 bits match the packet start marker.

#### Parameters

| Parameter | Accepted Type(s) | Default | Simple Explanation |
|---|---|---|---|
| `spectrum` | `np.ndarray` | Required | 2D complex real-FFT result. |
| `pairs_a` | `list[tuple[int, int]]` | Required | First coordinates for the pairs. |
| `pairs_b` | `list[tuple[int, int]]` | Required | Second coordinates for the pairs. |
| `sync_marker` | `bytes` | Required | Packet start bytes to compare.|

**Returns:** `bool`: `True` when the first 16 extracted bits match the marker; otherwise `False`.

In [13]:
import numpy as np
from spectral_watermark import SpectralEngine, BitCodec
e = SpectralEngine(); a, b = e.get_bin_pairs(16)
spectrum = np.fft.rfft2(np.zeros((64,64)))
print(e.check_sync_preamble(spectrum, a, b, BitCodec().sync_marker))

False


### SpectralEngine.sync_probe

**Signature:** `sync_probe(spectrum, a, b, sync)`

**What it does:** Performs the same 16-bit start-marker check using cached NumPy coordinate arrays.

#### Parameters

| Parameter | Accepted Type(s) | Default | Simple Explanation |
|---|---|---|---|
| `spectrum` | `np.ndarray` | Required | 2D complex real-FFT result. |
| `a` | `np.ndarray` | Required | Coordinate array shape `(n, 2)`. |
| `b` | `np.ndarray` | Required | Coordinate array shape `(n, 2)`. |
| `sync` | `bytes` | Required | Exact packet start bytes.|

**Returns:** `bool`: `True` when the start-marker probe matches; otherwise `False`.

In [14]:
import numpy as np
from spectral_watermark import SpectralEngine, BitCodec
e = SpectralEngine(); a, b = e.get_bin_arrays(16)
spectrum = np.fft.rfft2(np.zeros((64,64)))
print(e.sync_probe(spectrum, a, b, BitCodec().sync_marker))

False


## 5. Border Trimming & Scale Helpers (`calibration.py`)

The calibrator adds pilot peaks to each tile and estimates whether a screenshot was enlarged or reduced.

### PilotCalibrator.__init__

**Signature:** `PilotCalibrator(block_size=64, r_ref=16.0)`

**What it does:** Creates the pilot pattern and validates the tile and reference radius.

#### Parameters

| Parameter | Accepted Type(s) | Default | Simple Explanation |
|---|---|---|---|
| `block_size` | `int` | `64` | Positive even tile size. |
| `r_ref` | `float` | `16.0` | Reference pilot radius.|

**Returns:** A `PilotCalibrator` object. No value is returned explicitly.

In [15]:
from spectral_watermark import PilotCalibrator
c = PilotCalibrator()
print(type(c).__name__, c.block_size, c.r_ref)

PilotCalibrator 64 16.0


### PilotCalibrator.get_pilot_coords

**Signature:** `get_pilot_coords()`

**What it does:** Lists the two symmetric pilot positions reserved for scale detection.

#### Parameters

| Parameter | Accepted Type(s) | Default | Simple Explanation |
|---|---|---|---|
| None | — | — | This method takes no parameters.|

**Returns:** A `list[tuple[int, int]]` containing exactly two `(row, column)` coordinates.

In [16]:
from spectral_watermark import PilotCalibrator
coords = PilotCalibrator().get_pilot_coords()
print(coords, len(coords))

[(11, 11), (53, 11)] 2


### PilotCalibrator.inject_pilots

**Signature:** `inject_pilots(tile_spectrum, mean_magnitude)`

**What it does:** Strengthens two pilot pattern positions while keeping their existing directions.

#### Parameters

| Parameter | Accepted Type(s) | Default | Simple Explanation |
|---|---|---|---|
| `tile_spectrum` | `np.ndarray` | Required | 2D complex FFT spectrum to modify. |
| `mean_magnitude` | `float` | Required | Average pattern strength used to choose the boost.|

**Returns:** The same `np.ndarray` spectrum object, modified in place.

In [17]:
import numpy as np
from spectral_watermark import PilotCalibrator
c = PilotCalibrator(); spectrum = np.fft.rfft2(np.zeros((64,64)))
result = c.inject_pilots(spectrum, 10.0)
print(result.shape, result.dtype)

(64, 33) complex128


### PilotCalibrator.estimate_scale

**Signature:** `estimate_scale(cropped_image, search_radius_min=8.0, search_radius_max=28.0)`

**What it does:** Looks for pilot peaks and compares their spacing with the reference spacing to estimate zoom.

#### Parameters

| Parameter | Accepted Type(s) | Default | Simple Explanation |
|---|---|---|---|
| `cropped_image` | `np.ndarray` | Required | 2D grayscale or 3D RGB(A) image. |
| `search_radius_min` | `float` | `8.0` | Lowest pilot radius to inspect. |
| `search_radius_max` | `float` | `28.0` | Highest pilot radius to inspect.|

**Returns:** `float`: an estimated scale from `0.5` to `2.5`, or `1.0` when no reliable pilots are found.

In [18]:
import numpy as np
from spectral_watermark import PilotCalibrator
random = np.random.default_rng(1).integers(0,256,(64,64),dtype=np.uint8)
scale = PilotCalibrator().estimate_scale(random)
print(type(scale).__name__, round(scale, 3))

float 0.804


### trim_solid_borders

**Signature:** `trim_solid_borders(image, variance_threshold=12.0)`

**What it does:** Removes plain outer rows and columns from a screenshot without cropping detail-rich areas.

#### Parameters

| Parameter | Accepted Type(s) | Default | Simple Explanation |
|---|---|---|---|
| `image` | `np.ndarray` | Required | 2D grayscale or 3D `(H, W, 1/3/4)` image. |
| `variance_threshold` | `float` | `12.0` | Minimum variation needed to keep an edge row or column.|

**Returns:** A `np.ndarray`: a cropped copy for detail-rich borders, or the original array when trimming is unsafe.

In [19]:
import numpy as np
from spectral_watermark import trim_solid_borders
image = np.zeros((80,80), dtype=np.uint8); image[20:60,20:60] = 200
trimmed = trim_solid_borders(image)
print(image.shape, trimmed.shape)

(80, 80) (40, 40)


## 6. Command-Line Interface Reference (`cli.py`)

The installed entry point is `spectral-watermark`. Run `spectral-watermark --help` for the live command list.

### `spectral-watermark` — top-level CLI

**Signature:** `spectral-watermark [COMMAND] [OPTIONS]`

**What it does:** Opens the image-watermark command group and exposes `embed`, `extract`, and `benchmark`.

| Parameter / Flag | Accepted Value | Default | Simple Explanation |
|---|---|---|---|
| `--help` | Flag | Off | Show top-level commands and usage. |
| `--version` | Flag | Off | Print the package version. |
| `COMMAND` | `embed`, `extract`, or `benchmark` | Required | Choose the operation. |

**Returns:** A process exit status: `0` on success and a non-zero status on validation or recovery failure.

### `spectral-watermark embed`

**Signature:** `spectral-watermark embed -i INPUT -p PAYLOAD -o OUTPUT`

**What it does:** Reads a cover image, hides UTF-8 text, and saves a watermarked PNG or other supported image.

| Parameter / Flag | Accepted Value | Default | Simple Explanation |
|---|---|---|---|
| `-i`, `--input` | Existing image path | Required | Cover image file. |
| `-p`, `--payload` | UTF-8 string | Required | Message to hide; it must fit one tile. |
| `-o`, `--output` | Image path | Required | Destination file. Parent folders are created. |
| `--help` | Flag | Off | Show embed options. |

**Returns:** Prints dimensions, encoded bit length, PSNR, and elapsed time; exits `0` on success.

### `spectral-watermark extract`

**Signature:** `spectral-watermark extract -i INPUT [SCAN OPTIONS]`

**What it does:** Searches a watermarked image or crop for a valid text payload.

| Parameter / Flag | Accepted Value | Default | Simple Explanation |
|---|---|---|---|
| `-i`, `--input` | Existing image path | Required | Watermarked or cropped image. |
| `--stride` | Integer ≥ 1 | `2` | Pixel spacing for detailed search. |
| `--bit-len` | Integer ≥ 1 | None | Known encoded frame size in bits. |
| `--min-chars` | Integer ≥ 1 | `1` | Minimum allowed text length. |
| `--max-chars` | Integer ≥ 1 | `32` | Maximum allowed text length. |
| `--scale` | Positive float | None | Add an apparent zoom factor; repeatable. |
| `--multi-scale` | Flag | Off | Try preset factors `1.0`, `1.25`, `1.5`, `0.8`, and `0.75`. |
| `--timeout` | Positive seconds | None | Stop the search after this time. |
| `-q`, `--quiet` | Flag | Off | Print only recovered text. |
| `--help` | Flag | Off | Show extract options. |

**Returns:** Prints the recovered payload and elapsed time, or returns a Click error if none is found.

### `spectral-watermark benchmark`

**Signature:** `spectral-watermark benchmark -i INPUT [--payload TEXT] [--seed INTEGER]`

**What it does:** Embeds a test message, applies built-in image attacks, and reports whether each one still decodes.

| Parameter / Flag | Accepted Value | Default | Simple Explanation |
|---|---|---|---|
| `-i`, `--input` | Existing image path | Required | Cover image to test. |
| `-p`, `--payload` | UTF-8 string | `SECRET_2026` | Test message to embed and recover. |
| `--seed` | Integer | `42` | Seed for repeatable synthetic noise. |
| `--help` | Flag | Off | Show benchmark options. |

**Returns:** Prints baseline PSNR and an attack table; exits non-zero if baseline recovery fails.

### cli.main

**Signature:** `main()`

**What it does:** Runs the top-level Click command group when the package entry point starts.

#### Parameters

| Parameter | Accepted Type(s) | Default | Simple Explanation |
|---|---|---|---|
| None | — | — | This callable takes no explicit Python parameters.|

**Returns:** `None` on normal completion; Click may raise or set a non-zero process status on failure.

In [20]:
from spectral_watermark.cli import main
print(callable(main), type(main).__name__)

True Group
